# UMAP — comparaison des espaces de features

| Espace | Dimension | Description |
|--------|-----------|-------------|
| DINOv2 raw | 768 | CLS token ViT-B/14, backbone gelé |
| MSTCN features | 64 | Sortie du dernier stage TCN avant `output_proj` |
| LSTM features | 512 | Sortie LSTM bidirectionnel avant `output_proj` |
| TeCNO refinement | 32 | Dernier stage TCN de raffinement avant `output_proj` |

Phases inconnues **significatives** uniquement (Corneal_hydration exclue car trop courte et trop similaire à Wound_hydration).

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('../phases_recognition').resolve()))

import numpy as np
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf
from torch.utils.data import DataLoader
import umap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from dataset.feature_dataset import VideoFeatureDataset, _collate_single_video
from models import instantiate_model

print('Imports OK')

/home/helena/.conda/envs/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [ ]:
TEST_ROOT = '/home/helena/UCL_video_cataract/features_dino/test/'
MSTCN_EXP = '/home/helena/experiments_cataract/mstcn_dino_v1_date=2026_06_11_17_02_41'
LSTM_EXP  = '/home/helena/experiments_cataract/lstm_dino_v1_date=2026_06_11_17_40_27'
OOD_JSON  = '/home/helena/UCL_video_cataract/dataset_temporal/labels_ood.json'

CLASS_NAMES = [
    'Capsule_polishing', 'Hydrodissection', 'Incision',
    'Irrigation_and_aspiration', 'Lens_implant_settingup',
    'Phacoemulsification', 'Rhexis', 'Tonifying_and_antibiotics',
    'Viscous_agent_injection', 'Viscous_agent_removal', 'Wound_hydration',
]

# Phases connues : couleurs vives distinctes (pas pastel)
COLORS = [
    '#1f77b4',  # Capsule_polishing         - bleu
    '#2ca02c',  # Hydrodissection           - vert
    '#9467bd',  # Incision                  - violet
    '#8c564b',  # Irrigation_and_aspiration - marron
    '#e377c2',  # Lens_implant_settingup    - rose
    '#17becf',  # Phacoemulsification       - cyan
    '#bcbd22',  # Rhexis                    - olive
    '#d4a017',  # Tonifying_and_antibiotics - or
    '#f07800',  # Viscous_agent_injection   - orange
    '#3a7d44',  # Viscous_agent_removal     - vert forêt
    '#6b5b95',  # Wound_hydration           - violet foncé
]

EXCLUDE_UNKNOWN = {'Corneal_hydration'}
KNOWN_SET = set(CLASS_NAMES)

UNKNOWN_PHASES = [
    'Malyugin_ring_insertion', 'Malyugin_ring_removal',
    'Suture', 'Iris_manipulation', 'Trypan_blue_injection',
]
# Phases inconnues : couleurs néon vives, marqueur x (transparent au centre)
UNK_COLORS = {
    'Malyugin_ring_insertion': '#FF0000',
    'Malyugin_ring_removal':   '#FF8800',
    'Suture':                  '#FF00FF',
    'Iris_manipulation':       '#00CCFF',
    'Trypan_blue_injection':   '#00FF44',
}

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Charger les features DINOv2 + identifier les phases inconnues

In [3]:
import re

with open(OOD_JSON) as f:
    ood_data = json.load(f)

test_root  = pathlib.Path(TEST_ROOT)
feat_files = sorted(f for f in test_root.glob('*.npy')
                    if not f.stem.endswith('_labels') and not f.stem.endswith('_mahal'))

feat_list, label_list, unk_name_list = [], [], []

for feat_file in feat_files:
    label_file = test_root / f'{feat_file.stem}_labels.npy'
    if not label_file.exists():
        continue
    feats  = np.load(feat_file).astype(np.float32)
    labels = np.load(label_file).astype(np.int32)
    T = len(labels)

    # Extraire frame_num → phase pour TOUTES les frames de cette vidéo
    prefix = f'test/{feat_file.stem}/'
    frame_phase = {}  # {frame_num: phase}
    for key, phase in ood_data.items():
        if not key.startswith(prefix):
            continue
        m = re.search(r'Frame_(\d+)', key)
        if m:
            frame_phase[int(m.group(1))] = phase

    unk_names = np.full(T, '', dtype=object)

    if frame_phase:
        # Estimer le pas d'échantillonnage : max_frame_num / T
        sorted_frames = sorted(frame_phase.keys())
        max_frame = sorted_frames[-1]
        step = max(1, max_frame // T)

        # Pour chaque frame feature t, trouver la phase la plus proche
        for t in range(T):
            if labels[t] != -1:
                continue
            approx_frame = t * step
            # Frame la plus proche dans frame_phase
            closest = min(sorted_frames, key=lambda f: abs(f - approx_frame))
            unk_names[t] = frame_phase[closest]

    # Exclure Corneal_hydration
    corneal_mask = (labels == -1) & (unk_names == 'Corneal_hydration')
    labels[corneal_mask] = -2

    feat_list.append(feats)
    label_list.append(labels)
    unk_name_list.append(unk_names)

feats_dino    = np.concatenate(feat_list)
labels_all    = np.concatenate(label_list)
unk_names_all = np.concatenate(unk_name_list)

print(f'Total frames : {len(feats_dino):,}')
print(f'  Known   : {(labels_all >= 0).sum():,}')
print(f'  Unknown : {(labels_all == -1).sum():,}')
print(f'  Exclu (Corneal) : {(labels_all == -2).sum():,}')
print()
from collections import Counter
print('Phases inconnues détectées :')
print(Counter(unk_names_all[labels_all == -1]))

Total frames : 33,378
  Known   : 29,103
  Unknown : 4,156
  Exclu (Corneal) : 119

Phases inconnues détectées :
Counter({'Malyugin_ring_insertion': 1764, 'Malyugin_ring_removal': 592, 'Viscous_agent_injection': 497, 'Suture': 446, 'Irrigation_and_aspiration': 428, 'Incision': 219, 'Lens_implant_settingup': 53, 'Wound_hydration': 47, 'Tonifying_and_antibiotics': 42, 'Capsule_polishing': 40, 'Trypan_blue_injection': 28})


## 2. Extraire les features internes de MSTCN et LSTM

In [4]:
def load_model(exp_dir):
    cfg   = OmegaConf.load(f'{exp_dir}/config.yaml')
    model = instantiate_model(cfg.model)
    state = torch.load(f'{exp_dir}/ckpt/best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(state['model_state_dict'])
    return model.to(device).eval()

dataset = VideoFeatureDataset(root=TEST_ROOT)
loader  = DataLoader(dataset, batch_size=1, shuffle=False,
                     collate_fn=_collate_single_video)

print('Loading models...')
mstcn = load_model(MSTCN_EXP)
lstm  = load_model(LSTM_EXP)
print('Done.')

Loading models...
Done.


In [5]:
@torch.no_grad()
def extract_mstcn_features(model, loader):
    """64-dim : dernier stage TCN avant output_proj."""
    all_feats = []
    for features, _, _ in loader:
        _, f = model.forward_with_features(features.unsqueeze(0).to(device))
        all_feats.append(f.squeeze(0).T.cpu().numpy())
    return np.concatenate(all_feats)

@torch.no_grad()
def extract_with_hook(model, target_layer, loader, is_conv=False):
    """Extrait les features avant une couche via forward hook."""
    captured = []
    def hook(module, input, output):
        captured.append(input[0].detach().cpu())
    handle = target_layer.register_forward_hook(hook)
    all_feats = []
    for features, _, _ in loader:
        captured.clear()
        model(features.unsqueeze(0).to(device))
        t = captured[0]
        # Conv1d input: (B, C, T) → (T, C) ; Linear input: (B, T, C) → (T, C)
        if is_conv:
            all_feats.append(t.squeeze(0).T.numpy())
        else:
            all_feats.append(t.squeeze(0).numpy())
    handle.remove()
    return np.concatenate(all_feats)

print('MSTCN features (64-dim)...')
feats_mstcn = extract_mstcn_features(mstcn, loader)

print('LSTM features (512-dim)...')
feats_lstm = extract_with_hook(lstm, lstm.lstm_stage.output_proj, loader, is_conv=False)

print('TeCNO refinement features (32-dim)...')
feats_tecno = extract_with_hook(lstm, lstm.refinement_stages[-1].output_proj, loader, is_conv=True)

print(f'\nShapes: DINOv2={feats_dino.shape}, MSTCN={feats_mstcn.shape}, LSTM={feats_lstm.shape}, TeCNO={feats_tecno.shape}')

MSTCN features (64-dim)...
LSTM features (512-dim)...
TeCNO refinement features (32-dim)...

Shapes: DINOv2=(33378, 768), MSTCN=(33378, 64), LSTM=(33378, 512), TeCNO=(33378, 32)


## 3. Subsampling équilibré + UMAP

In [6]:
np.random.seed(42)
N_PER_CLASS  = 300   # frames par phase connue
N_PER_UNKNOWN = 400  # frames par type de phase inconnue

idx_keep = []

# Sous-échantillonnage équilibré par phase connue
for c in range(len(CLASS_NAMES)):
    idx_c = np.where(labels_all == c)[0]
    if len(idx_c) == 0:
        continue
    n = min(N_PER_CLASS, len(idx_c))
    idx_keep.append(np.random.choice(idx_c, n, replace=False))

# Sous-échantillonnage par type de phase inconnue
for pname in UNKNOWN_PHASES:
    idx_p = np.where((labels_all == -1) & (unk_names_all == pname))[0]
    if len(idx_p) == 0:
        continue
    n = min(N_PER_UNKNOWN, len(idx_p))
    idx_keep.append(np.random.choice(idx_p, n, replace=False))
    print(f'  {pname}: {n} frames')

idx_s    = np.sort(np.concatenate(idx_keep))
labels_s = labels_all[idx_s]
unk_names_s = unk_names_all[idx_s]

print(f'\nTotal subsample: {len(idx_s):,} frames')
print(f'  Known frames   : {(labels_s >= 0).sum():,}')
print(f'  Unknown frames : {(labels_s == -1).sum():,}')

spaces = {
    'DINOv2 (768-dim)':       feats_dino[idx_s],
    'MSTCN features (64-dim)':  feats_mstcn[idx_s],
    'LSTM features (512-dim)':  feats_lstm[idx_s],
    'TeCNO refine (32-dim)':    feats_tecno[idx_s],
}

print('\nComputing UMAP...')
embeddings = {}
for name, X in spaces.items():
    print(f'  {name}...', end=' ', flush=True)
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                        random_state=42, verbose=False)
    embeddings[name] = reducer.fit_transform(X)
    print('done')
print('All done!')

  Malyugin_ring_insertion: 400 frames
  Malyugin_ring_removal: 400 frames
  Suture: 400 frames
  Trypan_blue_injection: 28 frames

Total subsample: 4,528 frames
  Known frames   : 3,300
  Unknown frames : 1,228

Computing UMAP...
  DINOv2 (768-dim)... 

/home/helena/.conda/envs/venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
  MSTCN features (64-dim)... 

/home/helena/.conda/envs/venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
  LSTM features (512-dim)... 

/home/helena/.conda/envs/venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
  TeCNO refine (32-dim)... 

/home/helena/.conda/envs/venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


done
All done!


## 4. Visualisation

In [ ]:
# ── FIGURE 1 : Phases inconnues colorées par TYPE ──────────────────────────
def plot_umap_colored(ax, emb, labels, unk_names, title):
    # 1. Points connus (dessous)
    for c, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
        mask = labels == c
        if mask.sum() == 0: continue
        ax.scatter(emb[mask, 0], emb[mask, 1], c=color, s=18, alpha=0.75,
                   linewidths=0, zorder=2)
    # 2. Étoiles (dessus, légèrement transparentes pour voir dessous)
    mask_unk = labels == -1
    for pname, pcolor in UNK_COLORS.items():
        mask_p = mask_unk & (unk_names == pname)
        if mask_p.sum() == 0: continue
        ax.scatter(emb[mask_p, 0], emb[mask_p, 1], c=pcolor, s=200, alpha=0.75,
                   edgecolors='black', linewidths=0.5, marker='*', zorder=3)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.set_xticks([]); ax.set_yticks([])

fig1, axes1 = plt.subplots(2, 2, figsize=(28, 22))
axes1 = axes1.flatten()
for ax, (name, emb) in zip(axes1, embeddings.items()):
    plot_umap_colored(ax, emb, labels_s, unk_names_s, name)

fig1.suptitle('Figure 1 — Quel type de phase inconnue tombe où ?\n'
              '(chaque étoile = une frame inconnue, couleur = type de phase)',
              fontsize=18, y=1.01)

handles_known = [mpatches.Patch(color=c, label=n) for c, n in zip(COLORS, CLASS_NAMES)]
handles_unk   = [plt.scatter([], [], c=c, s=150, marker='*', edgecolors='black',
                              linewidths=0.5, label=n) for n, c in UNK_COLORS.items()]
leg1 = fig1.legend(handles=handles_known, loc='lower left', ncol=2, fontsize=13,
                   frameon=True, bbox_to_anchor=(0.01, -0.08), title='Known phases',
                   title_fontsize=14)
fig1.legend(handles=handles_unk, loc='lower right', ncol=1, fontsize=13,
            frameon=True, bbox_to_anchor=(0.99, -0.08), title='Unknown phases ★',
            title_fontsize=14)
fig1.add_artist(leg1)
plt.tight_layout()
out1 = '/home/helena/experiments_cataract/ood_detection/umap_colored_unknowns.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out1}')

In [ ]:
# ── FIGURE 2 : Phases connues UNIQUEMENT — structure des clusters ───────────
def plot_umap_known_only(ax, emb, labels, title):
    for c, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
        mask = labels == c
        if mask.sum() == 0: continue
        ax.scatter(emb[mask, 0], emb[mask, 1], c=color, s=18, alpha=0.75,
                   linewidths=0)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.set_xticks([]); ax.set_yticks([])

mask_known = labels_s >= 0

fig2, axes2 = plt.subplots(2, 2, figsize=(28, 22))
axes2 = axes2.flatten()
for ax, (name, emb) in zip(axes2, embeddings.items()):
    plot_umap_known_only(ax, emb[mask_known], labels_s[mask_known], name)

fig2.suptitle('Figure 2 — Structure des clusters (phases connues uniquement)',
              fontsize=18, y=1.01)

handles_known = [mpatches.Patch(color=c, label=n) for c, n in zip(COLORS, CLASS_NAMES)]
fig2.legend(handles=handles_known, loc='lower center', ncol=4, fontsize=13,
            frameon=True, bbox_to_anchor=(0.5, -0.05), title='Known phases',
            title_fontsize=14)
plt.tight_layout()
out2 = '/home/helena/experiments_cataract/ood_detection/umap_known_only.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out2}')

In [9]:
from sklearn.metrics import silhouette_score

mask_known = labels_s >= 0
print('Silhouette scores (higher = clusters plus séparés):\n')
scores = {}
for name, emb in embeddings.items():
    s = silhouette_score(emb[mask_known], labels_s[mask_known],
                         sample_size=2000, random_state=42)
    scores[name] = s
    print(f'  {name:<35}  {s:.4f}')

best = max(scores, key=scores.get)
print(f'\nMeilleur espace : {best}')

Silhouette scores (higher = clusters plus séparés):

  DINOv2 (768-dim)                     -0.1002
  MSTCN features (64-dim)              0.1111
  LSTM features (512-dim)              0.1255
  TeCNO refine (32-dim)                0.1275

Meilleur espace : TeCNO refine (32-dim)
